<a href="https://colab.research.google.com/github/mrayhanz/244107020027-Machine-Learning/blob/main/Minggu%206/MRayhanZamzami_JS06_ANN_P6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IDENTITAS DIRI

Nama : Muhammad Rayhan Zamzami

NIM : 244107020027

Kelas : TI-3G

Absen : 19

# PRAKTIKUM 6 - Penerapan ANN pada Rekomendasi Lagu (Dataset Spotify)


In [1]:
!pip install annoy faiss-cpu hnswlib scikit-learn pandas

In [2]:
import os
import urllib.request
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset Spotify
# -------------------------------
csv_path = 'dataset/spotify_songs.csv'
if not os.path.exists(csv_path):
    alt_path = 'spotify_songs.csv'
    if os.path.exists(alt_path):
        csv_path = alt_path
    else:
        print("Mengunduh dataset Spotify...")
        os.makedirs('dataset', exist_ok=True)
        url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv'
        urllib.request.urlretrieve(url, csv_path)

df = pd.read_csv(csv_path)
print("Ukuran dataset:", df.shape)
print(df[['track_name', 'track_artist', 'playlist_genre']].head())

features = ['danceability', 'energy', 'loudness', 'speechiness', 
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].dropna().values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')

k = 10  # jumlah nearest neighbors
print(f"Fitur audio ({len(features)} dimensi) berhasil distandarisasi untuk {len(X_scaled)} lagu.")

Ukuran dataset: (32833, 23)
                                          track_name      track_artist  \
0  I Don't Care (with Justin Bieber) - Loud Luxur...        Ed Sheeran   
1                    Memories - Dillon Francis Remix          Maroon 5   
2                    All the Time - Don Diablo Remix      Zara Larsson   
3                  Call You Mine - Keanu Silva Remix  The Chainsmokers   
4            Someone You Loved - Future Humans Remix     Lewis Capaldi   

  playlist_genre  
0            pop  
1            pop  
2            pop  
3            pop  
4            pop  
Fitur audio (9 dimensi) berhasil distandarisasi untuk 32833 lagu.


In [3]:
# -------------------------------
# Exact Nearest Neighbor (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled[:500])  # evaluasi sampel 500 query untuk efisiensi
time_exact = time.time() - start
print(f"Exact NN selesai dalam: {time_exact:.3f} s")

# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled[:500]]
time_annoy = time.time() - start
print(f"Annoy selesai dalam: {time_annoy:.3f} s")

# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled[:500], k=k)
time_hnsw = time.time() - start
print(f"HNSW selesai dalam: {time_hnsw:.3f} s")

# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
dist_faiss, idx_faiss = index_faiss.search(X_scaled[:500], k)
time_faiss = time.time() - start
print(f"FAISS IVF selesai dalam: {time_faiss:.3f} s")

Exact NN selesai dalam: 1.950 s


Annoy selesai dalam: 9.930 s
HNSW selesai dalam: 0.143 s


FAISS IVF selesai dalam: 0.189 s


In [4]:
# -------------------------------
# Contoh rekomendasi top-5 neighbors lagu pertama
# -------------------------------
first_song_name = df.iloc[0]['track_name']
first_song_artist = df.iloc[0]['track_artist']
print(f"=== Lagu Query: '{first_song_name}' oleh {first_song_artist} ===\n")

print("Top-5 Indeks Tetangga:")
print(f"Exact NN : {idx_exact[0][:5]}")
print(f"Annoy    : {idx_annoy[0][:5]}")
print(f"HNSW     : {idx_hnsw[0][:5]}")
print(f"FAISS    : {idx_faiss[0][:5]}")

# Tampilkan nama lagu hasil rekomendasi Exact NN
print("\n--- Rekomendasi Lagu Terkait (Exact NN) ---")
for r_idx in idx_exact[0][:5]:
    print(f"- {df.iloc[r_idx]['track_name']} ({df.iloc[r_idx]['track_artist']}) - Genre: {df.iloc[r_idx]['playlist_genre']}")

=== Lagu Query: 'I Don't Care (with Justin Bieber) - Loud Luxury Remix' oleh Ed Sheeran ===

Top-5 Indeks Tetangga:
Exact NN : [    0 29684 29734  4959 29713]
Annoy    : [29684, 0, 29734, 4959, 29713]
HNSW     : [29684     0 29734  4959 29713]
FAISS    : [    0 29684 29734  4959 29713]

--- Rekomendasi Lagu Terkait (Exact NN) ---
- I Don't Care (with Justin Bieber) - Loud Luxury Remix (Ed Sheeran) - Genre: pop
- I Don't Care (with Justin Bieber) - Loud Luxury Remix (Ed Sheeran) - Genre: edm
- I Miss You (feat. Julia Michaels) - Cahill Remix (Clean Bandit) - Genre: edm
- Name & Number (Shift K3Y) - Genre: pop
- Rise - TV Noise Ibiza Mix (Jonas Blue) - Genre: edm


In [5]:
# -------------------------------
# Evaluasi Recall@10 terhadap Exact NN
# -------------------------------
def calc_recall(preds, targets, k_eval=10):
    c = sum(len(set(preds[i][:k_eval]) & set(targets[i][:k_eval])) for i in range(len(preds)))
    return c / (len(preds) * k_eval)

rec_annoy = calc_recall(idx_annoy, idx_exact)
rec_hnsw = calc_recall(idx_hnsw, idx_exact)
rec_faiss = calc_recall(idx_faiss, idx_exact)

tabel_spotify = pd.DataFrame([
    {"Metode": "Exact NN (Brute-Force)", "Waktu Total (s)": f"{time_exact:.3f}", "Recall@10": "1.0000"},
    {"Metode": "Annoy (10 trees)", "Waktu Total (s)": f"{time_annoy:.3f}", "Recall@10": f"{rec_annoy:.4f}"},
    {"Metode": "HNSW (M=16, ef=200)", "Waktu Total (s)": f"{time_hnsw:.3f}", "Recall@10": f"{rec_hnsw:.4f}"},
    {"Metode": "FAISS IVF (nlist=100, nprobe=10)", "Waktu Total (s)": f"{time_faiss:.3f}", "Recall@10": f"{rec_faiss:.4f}"}
])
display(tabel_spotify)

,Metode,Waktu Total (s),Recall@10
0,Exact NN (Brute-Force),1.950,1.0000
1,Annoy (10 trees),9.930,0.9850
2,"HNSW (M=16, ef=200)",0.143,0.9850
3,"FAISS IVF (nlist=100, nprobe=10)",0.189,0.9884
